# Mel

## Tanítás

In [6]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape, UpSampling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# --- 1. MIXED PRECISION (T4/P100 GPU-n gyorsítás) ---
mixed_precision.set_global_policy('mixed_float16')

# --- ÚTVONALAK ---
H5_PATH           = "/content/spotify_dataset_compressed.h5"
SAVE_PATH_AE      = "/content/drive/MyDrive/Diplomamunka/spotify_autoencoder_mel_only.keras"
SAVE_PATH_ENCODER = "/content/drive/MyDrive/Diplomamunka/spotify_encoder_mel_only.keras"


# ==========================================
# 2. EGYÉNI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    """L2 normalizálás a bottleneck vektoron."""
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)


# ==========================================
# 3. DATA GENERATOR (CSAK MEL)
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, split_name='train', batch_size=64, shuffle=True, **kwargs):
        super().__init__(workers=4, use_multiprocessing=False, max_queue_size=10, **kwargs)

        self.h5_path    = h5_path
        self.batch_size = batch_size
        self.shuffle    = shuffle

        self.hf = h5py.File(self.h5_path, 'r')
        splits     = self.hf['ml/split'][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_split_indices  = np.arange(
            index * self.batch_size,
            min((index + 1) * self.batch_size, len(self.indices))
        )
        batch_global_indices = np.sort(self.indices[batch_split_indices])

        # CSAK a Mel spektrogramot olvassuk be
        X_mel = self.hf['spectrograms/mel'][batch_global_indices]
        X_mel = np.expand_dims(X_mel, axis=-1).astype(np.float32)

        # Mivel csak 1 bemenet és kimenet van, a Keras automatikusan lekezeli
        return X_mel, X_mel

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __del__(self):
        if hasattr(self, 'hf') and self.hf.id.valid:
            self.hf.close()


# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_encoder_branch(x, filters, pool_sizes, branch_name):
    current_tensor = x
    for i, f in enumerate(filters):
        current_tensor = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_enc_conv_{i+1}')(current_tensor)
        current_tensor = BatchNormalization(name=f'{branch_name}_enc_bn_{i+1}')(current_tensor)
        current_tensor = Activation('relu', name=f'{branch_name}_enc_relu_{i+1}')(current_tensor)
        current_tensor = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_enc_pool_{i+1}')(current_tensor)

    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(current_tensor)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(current_tensor)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])


def create_decoder_branch(latent_input, start_shape, upsample_sizes, filters, branch_name):
    x = Dense(int(np.prod(start_shape)), activation='relu', name=f'{branch_name}_dec_dense')(latent_input)
    x = Reshape(start_shape, name=f'{branch_name}_dec_reshape')(x)

    for i, up_size in enumerate(upsample_sizes):
        x = UpSampling2D(size=up_size, name=f'{branch_name}_dec_up_{i+1}')(x)
        x = Conv2DTranspose(filters[i], kernel_size=(3, 3), padding='same', name=f'{branch_name}_dec_deconv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_dec_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_dec_relu_{i+1}')(x)

    x = Conv2D(1, kernel_size=(3, 3), padding='same', activation='linear', dtype='float32', name=f'{branch_name}_output')(x)
    return x


# ==========================================
# 5. MODELL ÖSSZEÁLLÍTÁSA (CSAK MEL)
# ==========================================
print("\n🚀 Autoencoder építése (Mel-Only)...")

input_mel = Input(shape=(128, 1280, 1), name='mel_input')

# Encoder
branch_mel = create_encoder_branch(input_mel, [32, 64, 128, 256], [(2, 4), (2, 4), (2, 4), (2, 2)], "mel")

# Nincs Concat, mehet egyenesen a sűrű rétegbe!
z = Dense(512, activation='relu', name='dense_1')(branch_mel)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

# Bottleneck
bottleneck_raw = Dense(256, activation='relu', dtype='float32', name='bottleneck')(z)
bottleneck     = L2NormLayer(name='l2_norm')(bottleneck_raw)

# Decoder
dec_mel = create_decoder_branch(bottleneck, start_shape=(8, 10, 256), upsample_sizes=[(2, 2), (2, 4), (2, 4), (2, 4)], filters=[128, 64, 32, 16], branch_name="mel")

autoencoder = Model(
    inputs=input_mel,
    outputs=dec_mel,
    name='spotify_autoencoder_mel_only'
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

encoder = Model(
    inputs=input_mel,
    outputs=bottleneck,
    name='encoder_mel_only'
)

# A summary-t kivettem, ahogy kérted!

# ==========================================
# 6. TANÍTÁS
# ==========================================
print("\n⚙️ Generátorok inicializálása...")

BATCH_SIZE = 64

train_gen = SpotifyDataGenerator(H5_PATH, split_name='train', batch_size=BATCH_SIZE)
val_gen   = SpotifyDataGenerator(H5_PATH, split_name='val',   batch_size=BATCH_SIZE, shuffle=False)

callbacks = [
    ModelCheckpoint(SAVE_PATH_AE, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print(f"\n🔥 GPU TANÍTÁS INDÍTÁSA... (Batch size: {BATCH_SIZE})")
history = autoencoder.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

# ==========================================
# 7. ENCODER MENTÉSE
# ==========================================
encoder.save(SAVE_PATH_ENCODER)
print(f"\n✅ Encoder elmentve: {SAVE_PATH_ENCODER}")


🚀 Autoencoder építése (Mel-Only)...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

🔥 GPU TANÍTÁS INDÍTÁSA... (Batch size: 64)
Epoch 1/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0891 - mae: 0.1610
Epoch 1: val_loss improved from None to 0.03510, saving model to /content/drive/MyDrive/Diplomamunka/spotify_autoencoder_mel_only.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_autoencoder_mel_only.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 798s 2s/step - loss: 0.0290 - mae: 0.1096 - val_loss: 0.0351 - val_mae: 0.1530 - learning_rate: 0.0010
Epoch 2/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0133 - mae: 0.0898
Epoch 2: val_loss improved from 0.03510 to 0.01340, saving model to /content/drive/MyDrive/Diplomamunka/spotify_autoencoder_mel_only.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_autoencoder_mel_only.keras
339/339 ━━━

## Kiértékelés

In [1]:
import os
import h5py
import numpy as np
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
import random

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" 
AE_VECTORS_PATH = "../Models/ae_vectors_closed_world_mel_only.npy"

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500]
NUM_TEST_SAMPLES = 1000
STEP_SIZE = 2

# ==========================================
# 1. SEGÉDFÜGGVÉNYEK
# ==========================================
def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def calculate_auc_fast(sims_array, relevant_indices, valid_indices, num_negatives=1000):
    pos_scores = [sims_array[idx] for idx in relevant_indices]
    if not pos_scores: return 0.5
        
    possible_negatives = list(set(valid_indices) - set(relevant_indices))
    n_neg = min(num_negatives, len(possible_negatives))
    
    neg_scores = []
    if n_neg > 0:
        chosen_negatives = random.sample(possible_negatives, n_neg)
        neg_scores = [sims_array[idx] for idx in chosen_negatives]
    
    if not neg_scores: return 0.5
        
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 2. FŐPROGRAM
# ==========================================
def main():
    print("1. Word2Vec és AE vektorok betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    
    ae_vectors = np.load(AE_VECTORS_PATH)
    print(f" -> Autoencoder vektorok betöltve. Alakja: {ae_vectors.shape}")

    print("\n2. Zárt Világ (Closed World) halmaz meghatározása...")
    with h5py.File(H5_PATH, "r") as hf:
        all_uris_bytes = hf["ml/track_uri"][:]
        all_uris = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in all_uris_bytes])
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])

    w2v_vocab = set(w2v_model.wv.key_to_index.keys())
    valid_uris = set(all_uris).intersection(w2v_vocab)
    
    uri_to_index = {uri: i for i, uri in enumerate(all_uris)}
    valid_indices = [uri_to_index[uri] for uri in valid_uris]
    
    print(f" -> Érvényes dalok a Zárt Világban: {len(valid_uris):,} db")

    print("\n3. Nyers (Raw) Akusztikus adatbázis legenerálása (Csak Mel)...")
    raw_vectors_list = []
    
    with h5py.File(H5_PATH, "r") as hf:
        batch_size = 128 
        for i in tqdm(range(0, len(all_uris), batch_size), desc="Nyers jellemzők kinyerése"):
            mel_batch = hf["spectrograms/mel"][i:i+batch_size]
            
            # Átlagolás az idődimenzió mentén
            mel_raw = np.mean(mel_batch, axis=1)
            
            # Mivel csak Mel van, nincs szükség összefűzésre (concatenate). 
            # Egyből ezt normalizáljuk L2 normával:
            raw_norms = np.linalg.norm(mel_raw, axis=1, keepdims=True)
            raw_normed = np.divide(mel_raw, raw_norms, out=np.zeros_like(mel_raw), where=raw_norms!=0)
            
            raw_vectors_list.extend(raw_normed)

    raw_vectors = np.array(raw_vectors_list, dtype=np.float32)

    w2v_matrix = np.zeros((len(all_uris), w2v_model.vector_size), dtype=np.float32)
    for i, uri in enumerate(all_uris):
        if uri in w2v_model.wv:
            w2v_matrix[i] = w2v_model.wv[uri]

    print("\n4. Ground Truth szótárak építése...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)

    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        gt_uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            u = gt_uris[i].decode('utf-8') if isinstance(gt_uris[i], bytes) else gt_uris[i]
            track_to_playlists[u].add(pid)
            playlist_to_tracks[pid].add(u)

    print("\n5. Teszt halmaz kinyerése...")
    test_indices = [i for i, s in enumerate(splits_str) if s == "test" and all_uris[i] in valid_uris]
    test_indices = test_indices[::STEP_SIZE]
    
    if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
        test_indices = test_indices[:NUM_TEST_SAMPLES]

    results = {
        "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "raw":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "ae":       {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
    }

    print("\n6. Kiértékelés futtatása (Fair, Closed-World mód)...")
    for idx in tqdm(test_indices, desc="Dalok tesztelése"):
        query_uri = all_uris[idx]
        
        pids = track_to_playlists.get(query_uri, set())
        if not pids: continue
            
        relevant_uris = set()
        for pid in pids:
            relevant_uris.update(playlist_to_tracks[pid])
        relevant_uris.discard(query_uri)
        relevant_uris = relevant_uris.intersection(valid_uris)
        
        if not relevant_uris: continue
        
        relevant_indices = [uri_to_index[u] for u in relevant_uris]

        # --- A) BASELINE (Word2Vec) ---
        query_w2v = w2v_matrix[idx].reshape(1, -1)
        sims_w2v = cosine_similarity(query_w2v, w2v_matrix)[0]
        sorted_indices_w2v = np.argsort(sims_w2v)[::-1]
        
        w2v_recs = [all_uris[i] for i in sorted_indices_w2v if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["baseline"]["auc"].append(calculate_auc_fast(sims_w2v, relevant_indices, valid_indices))

        # --- B) NYERS AUDIÓ BASELINE (RAW) ---
        query_raw = raw_vectors[idx].reshape(1, -1)
        sims_raw = cosine_similarity(query_raw, raw_vectors)[0]
        sorted_indices_raw = np.argsort(sims_raw)[::-1]
        
        raw_recs = [all_uris[i] for i in sorted_indices_raw if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["raw"]["auc"].append(calculate_auc_fast(sims_raw, relevant_indices, valid_indices))

        # --- C) AUTOENCODER KERESŐMOTOR ---
        query_ae = ae_vectors[idx].reshape(1, -1)
        sims_ae = cosine_similarity(query_ae, ae_vectors)[0]
        sorted_indices_ae = np.argsort(sims_ae)[::-1]
        
        ae_recs = [all_uris[i] for i in sorted_indices_ae if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["ae"]["auc"].append(calculate_auc_fast(sims_ae, relevant_indices, valid_indices))

        # --- METRIKÁK KISZÁMÍTÁSA ---
        for k in K_VALUES:
            hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
            results["baseline"][k]["hr"].append(hr_b)
            results["baseline"][k]["prec"].append(prec_b)
            results["baseline"][k]["rec"].append(rec_b)
            
            hr_r, prec_r, rec_r = calculate_metrics(raw_recs, relevant_uris, k)
            results["raw"][k]["hr"].append(hr_r)
            results["raw"][k]["prec"].append(prec_r)
            results["raw"][k]["rec"].append(rec_r)

            hr_ae, prec_ae, rec_ae = calculate_metrics(ae_recs, relevant_uris, k)
            results["ae"][k]["hr"].append(hr_ae)
            results["ae"][k]["prec"].append(prec_ae)
            results["ae"][k]["rec"].append(rec_ae)

    # --- 7. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*75)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆")
    print("="*75)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):")
    print(f"  1. BASELINE (Word2Vec): {np.mean(results['baseline']['auc'])*100:05.2f}%")
    print(f"  2. NYERS AUDIÓ (Raw):   {np.mean(results['raw']['auc'])*100:05.2f}%")
    print(f"  3. AUTOENCODER (AE):    {np.mean(results['ae']['auc'])*100:05.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("1. BASELINE (Word2Vec / Lejátszási Listák):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:05.2f}%")
        
        print("2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):")
        print(f"  Hit Rate@{k}:  {np.mean(results['raw'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['raw'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['raw'][k]['rec'])*100:05.2f}%")

        print("3. AUTOENCODER (Mélytanulásos Akusztika):")
        print(f"  Hit Rate@{k}:  {np.mean(results['ae'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['ae'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['ae'][k]['rec'])*100:05.2f}%")
    print("\n" + "="*75)

if __name__ == "__main__":
    main()

1. Word2Vec és AE vektorok betöltése...
 -> Autoencoder vektorok betöltve. Alakja: (27052, 256)

2. Zárt Világ (Closed World) halmaz meghatározása...
 -> Érvényes dalok a Zárt Világban: 25,511 db

3. Nyers (Raw) Akusztikus adatbázis legenerálása (Csak Mel)...


Nyers jellemzők kinyerése: 100%|██████████| 212/212 [00:40<00:00,  5.21it/s]



4. Ground Truth szótárak építése...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [03:04<00:00, 359711.01it/s]



5. Teszt halmaz kinyerése...

6. Kiértékelés futtatása (Fair, Closed-World mód)...


Dalok tesztelése: 100%|██████████| 1000/1000 [07:32<00:00,  2.21it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):
  1. BASELINE (Word2Vec): 76.84%
  2. NYERS AUDIÓ (Raw):   48.76%
  3. AUTOENCODER (AE):    56.76%

--- TOP 1 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@1:  87.10% | Prec: 87.10% | Rec: 00.05%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@1:  21.90% | Prec: 21.90% | Rec: 00.01%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@1:  36.20% | Prec: 36.20% | Rec: 00.01%

--- TOP 5 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@5:  96.30% | Prec: 79.28% | Rec: 00.22%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@5:  52.90% | Prec: 17.94% | Rec: 00.02%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@5:  71.60% | Prec: 32.46% | Rec: 00.05%

--- TOP 10 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@10:  97.70% | Prec: 76.19% | Rec: 00.41%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rat